# Phase 0: Setup & Foundations

**Enterprise Agentic RAG System**  
Tech Stack: **LlamaIndex** + **Ollama (Gemma)** + Local Embeddings + Ragas

This notebook validates the core foundations:
- Configuration system (YAML + Pydantic)
- Structured logging
- Ollama connectivity (LLM + Embeddings)
- LlamaIndex global Settings configuration
- PDF ingestion using LlamaIndex readers
- Basic node parsing / chunking

Run this after installing dependencies and pulling the required Ollama models.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# Ensure src is importable when running from notebooks/
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

## 1. Load Configuration & Logging

In [ ]:
from src.config import get_settings, settings
from src.logging_config import logger, setup_logging

print("Environment:", settings.rag_env)
print("Ollama base URL:", settings.ollama.base_url)
print("LLM model:", settings.ollama.llm_model)
print("Embedding model:", settings.ollama.embed_model)
print("Chunk size:", settings.llama_index.chunk_size)
print("Primary document:", settings.document.primary_pdf)

In [ ]:
setup_logging()
logger.info("Phase 0 setup notebook started")

## 2. Configure LlamaIndex with Ollama (Critical Step)

In [ ]:
# This applies the config.yaml values to LlamaIndex's global Settings
settings.configure_llama_index()

from llama_index.core import Settings as LlamaSettings

print("LlamaIndex LLM:", LlamaSettings.llm)
print("LlamaIndex Embed Model:", LlamaSettings.embed_model)
print("Chunk size / overlap:", LlamaSettings.chunk_size, "/", LlamaSettings.chunk_overlap)

## 3. Verify Ollama Connectivity

In [ ]:
import ollama

client = ollama.Client(host=settings.ollama.base_url)

print("Checking Ollama models...")
models = client.list()

model_names = [m.model for m in models.models]
print("Available models:", model_names)

required = [settings.ollama.llm_model, settings.ollama.embed_model]
for r in required:
    if r not in model_names:
        print(f"⚠️  WARNING: Model '{r}' is not pulled yet. Run: ollama pull {r}")
    else:
        print(f"✅ {r} is available")

## 4. Load the AI Agents Guidebook using LlamaIndex

In [ ]:
from llama_index.core import SimpleDirectoryReader
from pathlib import Path

raw_dir = Path(settings.paths.resolve()["data_raw"])
pdf_path = raw_dir / settings.document.primary_pdf

print(f"Loading document from: {pdf_path}")

reader = SimpleDirectoryReader(
    input_files=[str(pdf_path)],
    filename_as_id=True,
)

documents = reader.load_data()

print(f"✅ Loaded {len(documents)} document(s)")
print(f"Total characters (approx): {sum(len(d.text) for d in documents):,}")

# Show metadata from first page
if documents:
    print("\nFirst document metadata:")
    print(documents[0].metadata)
    print("\nSample text (first 600 chars):")
    print(documents[0].text[:600])

## 5. Node Parsing / Chunking with LlamaIndex

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

parser = SentenceSplitter(
    chunk_size=settings.llama_index.chunk_size,
    chunk_overlap=settings.llama_index.chunk_overlap,
)

nodes = parser.get_nodes_from_documents(documents)

print(f"✅ Created {len(nodes)} nodes/chunks")
print(f"Average node length (chars): {sum(len(n.text) for n in nodes) / max(1, len(nodes)):.0f}")

# Inspect a couple of nodes
for i, node in enumerate(nodes[:2]):
    print(f"\n--- Node {i} (page {node.metadata.get('page_label', '?')}) ---")
    print(node.text[:400])
    print("...")

## 6. (Optional) Quick Vector Index Smoke Test

Uncomment to create a small in-memory index and run a query.
This will call your local embedding model.

In [ ]:
# from llama_index.core import VectorStoreIndex

# print("Building small VectorStoreIndex (first 40 nodes only for speed)...")
# small_index = VectorStoreIndex(nodes[:40])

# query_engine = small_index.as_query_engine(
#     similarity_top_k=settings.llama_index.similarity_top_k,
#     response_mode=settings.llama_index.response_mode,
# )

# response = query_engine.query(
#     "What are the core principles of building reliable AI agents according to the guidebook?"
# )

# print("\nQuery Response:")
# print(response)

## Next Steps After This Notebook

1. Make sure Ollama is running and the required models are pulled:
   ```bash
   ollama pull gemma2:27b
   ollama pull nomic-embed-text
   ```
2. Proceed to build the ingestion pipeline using LlamaIndex readers + node parsers.
3. Create persistent vector stores (Chroma via LlamaIndex).
4. Implement retrieval + generation pipelines.
5. Add Ragas evaluation harness.

Phase 0 foundation is solid. Ready for agentic patterns in later phases.